<a href="https://colab.research.google.com/github/Bosmithan/Code/blob/main/BigramLangueModle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import  the required Libraries

In [ ]:
import math
import os
from collections import Counter, defaultdict
import nltk
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from google.colab import drive

Mount the Google drive

In [ ]:
print("Mounting Google Drive...")
drive.mount('/content/drive')


Mounting Google Drive...
Mounted at /content/drive


Tokenization and Preprossing

In [ ]:
try:
    nltk.data.find("tokenizers/punkt")
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt")
    nltk.download("punkt_tab")

def preprocess_text(text):
    lines = text.strip().split("\n")
    processed_lines = []
    for line in lines:
        if line.strip():
            # Lowercase and tokenize the sentence
            tokens = word_tokenize(line.lower())
            processed_lines.append(tokens)
    return processed_lines


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Build the Bigram Langue Modle

In [ ]:
def train_bigram_model(tokenized_lines):
    bigram_counts = defaultdict(Counter)
    unigram_counts = Counter()
    vocabulary = set()

    for tokens in tokenized_lines:
        if not tokens:
            continue

        # Add sentence boundary markers
        tokens = ["<s>"] + tokens + ["</s>"]

        for i in range(len(tokens) - 1):
            w1, w2 = tokens[i], tokens[i + 1]
            unigram_counts[w1] += 1
            bigram_counts[w1][w2] += 1
            vocabulary.add(w1)
            vocabulary.add(w2)

        # Count the last token
        unigram_counts[tokens[-1]] += 1
        vocabulary.add(tokens[-1])

    return bigram_counts, unigram_counts, vocabulary

def calculate_line_log_probability(tokens, bigram_counts, unigram_counts, total_vocab_size):
    tokens = ["<s>"] + tokens + ["</s>"]
    log_prob = 0.0

    for i in range(len(tokens) - 1):
        w1, w2 = tokens[i], tokens[i + 1]

        # Add-one smoothing formula:
        # P(w2 | w1) = (count(w1, w2) + 1) / (count(w1) + V)
        count_w1_w2 = bigram_counts[w1][w2]
        count_w1 = unigram_counts[w1]

        prob = (count_w1_w2 + 1) / (count_w1 + total_vocab_size)
        log_prob += math.log(prob)

    return log_prob

load data files(langue data) from Google Drive

In [ ]:
def main():
    base_path = "/content/drive/My Drive/CSC4182_lab"

    files = {
        "English": os.path.join(base_path, "LangId.train.English"),
        "French": os.path.join(base_path, "LangId.train.French"),
        "Italian": os.path.join(base_path, "LangId.train.Italian"),
    }
    test_file_path = os.path.join(base_path, "LangId.test")

    if not os.path.exists(test_file_path):
        raise FileNotFoundError(f"Could not find files at {base_path}. Please check your Google Drive folder structure.")

    raw_train_data = {}
    for lang, path in files.items():
        with open(path, "r", encoding="latin-1") as f:
            raw_train_data[lang] = f.read()

    with open(test_file_path, "r", encoding="latin-1") as f:
        test_text = f.read()


Apply the Bigram Langue modle